# Two-neuron excitatory/inhibitory source-to-field/readout workflow

This tutorial extends the single-neuron workflow to a minimal two-neuron excitatory/inhibitory (E/I) network.

**What you will learn:**
- Configure a two-neuron E/I network
- Simulate excitatory and inhibitory neuron dynamics
- Apply the eight standard proxy readout operators to a small network
- Inspect the output bundle with recurrent activity
- Understand the computational scope and readout framework for coupled networks

**Prerequisites:** Familiarity with [Tutorial 1: Single-neuron multimodal](01_single_neuron_multimodal.ipynb)

**Estimated time:** 5–10 minutes

In [ ]:
!pip install jaxfne

In [ ]:
import jaxfne
print(f"jaxfne version: {jaxfne.__version__}")

## Imports

Import JAX and jaxfne components.

In [ ]:
import jax
import jax.numpy as jnp
from jaxfne.fields import (
    spk_probe,
    vm_probe,
    source_probe,
    lfp_proxy_probe,
    csd_proxy_probe,
    eeg_proxy_probe,
    meg_proxy_probe,
    emm_proxy_probe,
)

# Verify JAX is available
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## Configuration

Set up a two-neuron network with one excitatory and one inhibitory Izhikevich neuron.

In [ ]:
# Create two-neuron E/I configuration
# One excitatory, one inhibitory, 100ms simulation, 0.1ms dt
cfg = (
    jaxfne.configuration()
    .network(
        name="two_neuron_ei",
        kind="coupled_neurons",
        n=2,
        cell_types={"E": 0.5, "I": 0.5},
    )
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(
        domain="laminar_column",
        conductivity="proxy",
        boundary="declared_proxy",
        gauge="mean_zero",
    )
    .probe(
        name="multimodal_two_neuron_ei",
        modes=[
            "spikes",
            "V_m",
            "source",
            "phi_e",
            "J_e",
            "CSD",
            "LFP",
        ],
    )
)

# Construct the JAX-native model
model = jaxfne.construct(cfg)
print("Two-neuron E/I model constructed successfully.")

## Run Simulation

Simulate 100 milliseconds of E/I network activity.
The two neurons interact through the shared field and dynamics.

In [ ]:
# Define simulation parameters
sim = jaxfne.simulation(duration_ms=100.0, dt_ms=0.1, seed=42)

# Run the simulation
signals = model.simulate(sim)

print(f"Simulation complete.")
print(f"Spike matrix shape: {signals.spikes.shape}  [time, neurons]")
print(f"Voltage shape: {signals.V_m.shape}         [time, neurons]")
if hasattr(signals, 'field') and signals.field is not None:
    print(f"LFP proxy shape: {signals.field.lfp_proxy.shape}    [time, contacts]")

## Apply Readout Operators

Apply all eight standard proxy readout operators to the two-neuron network signals.
These operators extract different modalities and summary statistics from the network activity.

In [ ]:
# Apply all eight probe operators
spk_readout = spk_probe(signals.spikes)
vm_readout = vm_probe(signals.V_m)

# Source and field-derived operators
source_readout = (
    source_probe(signals.sources[0]) if signals.sources is not None else None
)
lfp_readout = (
    lfp_proxy_probe(signals.field.lfp_proxy)
    if signals.field is not None
    else None
)
csd_readout = (
    csd_proxy_probe(signals.field.csd_proxy)
    if signals.field is not None
    else None
)

# EEG, MEG, EMM operators use field signal
field_signal = (
    signals.field.lfp_proxy if signals.field is not None else signals.V_m
)
eeg_readout = eeg_proxy_probe(field_signal)
meg_readout = meg_proxy_probe(field_signal)
emm_readout = emm_proxy_probe(field_signal)

# Construct probe_report from all operators
probe_report = {
    "spk": spk_readout.report,
    "vm": vm_readout.report,
}
if source_readout is not None:
    probe_report["source"] = source_readout.report
if lfp_readout is not None:
    probe_report["lfp_proxy"] = lfp_readout.report
if csd_readout is not None:
    probe_report["csd_proxy"] = csd_readout.report

probe_report.update({
    "eeg_proxy": eeg_readout.report,
    "meg_proxy": meg_readout.report,
    "emm_proxy": emm_readout.report,
})

print(f"All eight probe operators applied to two-neuron network:")
for op_name in probe_report.keys():
    print(f"  ✓ {op_name}")

## Inspect Network Activity

Examine spike and voltage patterns from both neurons to see E/I dynamics.

In [ ]:
# Analyze per-neuron activity
spike_counts = jnp.sum(signals.spikes, axis=0)  # spikes per neuron
voltage_means = jnp.mean(signals.V_m, axis=0)   # mean voltage per neuron
voltage_stds = jnp.std(signals.V_m, axis=0)     # std voltage per neuron

print("Per-neuron activity:")
for i in range(2):
    neuron_type = "Excitatory" if i == 0 else "Inhibitory"
    print(f"  Neuron {i} ({neuron_type}):")
    print(f"    Spikes: {int(spike_counts[i])}")
    print(f"    Voltage mean: {float(voltage_means[i]):.2f} mV")
    print(f"    Voltage std: {float(voltage_stds[i]):.2f} mV")

## Output Bundle Inspection

Examine the reproducible output bundle: manifest (metadata) and probe reports.

In [ ]:
# Get the manifest (full pipeline metadata)
manifest = model.manifest(signals=signals)

# Compute basic network metrics
spk_rate = float(jnp.mean(jnp.sum(signals.spikes, axis=1)))
vm_mean = float(jnp.mean(signals.V_m))
vm_std = float(jnp.std(signals.V_m))

metrics = {
    "network_spike_rate_per_timestep": spk_rate,
    "network_Vm_mean_mV": vm_mean,
    "network_Vm_std_mV": vm_std,
}

# Display manifest keys
print("Manifest keys (pipeline metadata):")
manifest_keys = list(manifest.keys())[:5]
for key in manifest_keys:
    print(f"  - {key}")
if len(manifest.keys()) > 5:
    print(f"  ... and {len(manifest.keys()) - 5} more")

# Display network metrics
print(f"\nNetwork metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.6f}")

## Readout Summary

Verify the scope metadata from the manifest.
These fields indicate the computational framework and any validation present.
All readouts remain computational proxies: no biological mechanism claims, no empirical validation.

In [ ]:
# Extract and display scope metadata
scope_summary = {
    "claim_level": manifest.get("claim_level"),
    "field_claim_level": manifest.get("field_claim_level"),
    "field_solver_status": manifest.get("field_solver_status"),
    "source_calibration_status": manifest.get("source_calibration_status"),
    "physical_amplitude_claim_allowed": manifest.get("physical_amplitude_claim_allowed"),
}

print(f"Readout summary (immutable, computational framework):")
for key, value in scope_summary.items():
    print(f"  {key}: {value}")

print(f"\n✓ Physical amplitude claims allowed: {scope_summary['physical_amplitude_claim_allowed']}")
print(f"✓ All eight operators executed successfully.")
print(f"✓ Output bundle is JSON-serializable and reproducible.")
print(f"\nScope: Computational scaffold for E/I network learning, not empirically validated.")

## Summary

You have completed the two-neuron E/I multimodal source-to-field/readout workflow:

1. **Configuration:** Defined a two-neuron network (one excitatory, one inhibitory) with Izhikevich emitters and laminar field
2. **Simulation:** Ran 100ms of JAX-native E/I network dynamics (1000 timesteps)
3. **Readouts:** Applied eight standard proxy operators to network signals
4. **Inspection:** Examined per-neuron activity, manifest, metrics, and scope metadata

### What's next

- **[Tutorial 3: 100-neuron E/I network](03_network_100_ei.md)** — Scale to population dynamics
- **[Guides: Probe operators](../guides/probe_operators.md)** — Details on each readout operator
- **[API Reference](../api/index.md)** — Full function and class documentation
- **[Tensor-field workflows](../guides/tensor_field_workflows.md)** — Deep dive on field mathematics

### Questions?

File an issue or discuss on GitHub: https://github.com/HNXJ/jaxfne